# Study 1: Clinical performance

CTRS scores and general clinical performance dimensions.

In [26]:
import random
import pandas as pd
pd.set_option('future.no_silent_downcasting', True)

import scipy.stats as sp
import statsmodels.api as sm
import statsmodels.formula.api as smf

import plotly.express as px
import plotly.graph_objects as go

from utils.utils import *
from utils.variables import *

## Read in data

Only analyze transcripts that expert consortium rated as being appropriate for CBT.

In [27]:
# Filter out transcripts that are not appropriate for CBT
df_pids, df_ctrs, df_general, df_comparisons = retrieve_analysed_data('clinicians')

Removing 7 transcripts that are not appropriate for CBT
N = 227


condition        model 
cognitive_layer  claude    25
                 gemini    26
                 gpt4      26
                 llama     24
human_therapist  human     26
standalone_llms  claude    25
                 gemini    24
                 gpt4      27
                 llama     24
dtype: int64

## CTRS

### Overall CTRS score

Averaged across the 11 subscales.

In [28]:
# Overall CTRS score per transcript
df = (
        df_ctrs[['pid','item','human1_score']]
        .copy()
        .rename(columns={'human1_score':'score'})
        .groupby('pid')['score']
        .mean()
        .reset_index()
    )

# Add transcript metadata
df = df.merge(df_pids[['pid','condition','model']], on='pid', how='left')

print(f"N = {len(df.dropna())}")

# Add control variables (expert's "belief in humanness" of the reviewed agent)
control = df_general.loc[df_general['item']=='was_human',['pid','score']].copy().reset_index(drop=True)
control = control.rename(columns={'score':'believed_human'})

df = df.merge(control, on='pid', how='left')

# Set columns to the correct type
df['score'] = df['score'].astype(float)
df['believed_human'] = df['believed_human'].replace({
    'Strong disagree': 0,
    'Disagree': 1,
    'Neutral': 2,
    'Agree': 3,
    'Strong agree': 4
}).astype(float)

# Print summary of the data
print("OVERALL CTRS:")
display(df.groupby('condition')['score'].agg(['mean','std']).round(2))

# Print summary of the "believed_human" variable
df['believed_human_percent'] = df['believed_human'] / 4
print("BELIEVED HUMAN:")
display(
    df.groupby('condition')['believed_human_percent']
    .agg(['mean','std'])
    .map(lambda x: f"{x*100:.1f}%")
)

# ------------------------------------------------------------------------------------------------------------------------------------
# Run models
# ------------------------------------------------------------------------------------------------------------------------------------

# --- 2 x 4 model
model_df = df.loc[df['condition']!='human_therapist',].copy().reset_index(drop=True)

model_fit = smf.ols('score ~ C(model) * C(condition) * believed_human', data=model_df).fit()

display_residuals(model_fit)

anova_table = format_anova_table(model_fit)

print_title('Overall CTRS: 2 x 4 model')
print(anova_table)

# --- 3-way model
model_df = df.groupby(['pid','condition','believed_human'])['score'].mean().reset_index()

model_fit = smf.ols('score ~ C(condition) * believed_human', data=model_df).fit()

display_residuals(model_fit)

anova_table = format_anova_table(model_fit)

print_title('Overall CTRS: 3-way model')
print(anova_table)

# ------------------------------------------------------------------------------------------------------------------------------------
# Compare to humans
# ------------------------------------------------------------------------------------------------------------------------------------
print_title('Human Benchmarking')

m_diff = df.loc[df['condition']=='cognitive_layer','score'].mean() - df.loc[df['condition']=='human_therapist','score'].mean()
print(f"Cognitive Layer - Human Therapists = {m_diff:.1f}")
print(anova_pairwise_comparisons(model_fit))

human_benchmark = df.loc[df['condition']=='human_therapist','score'].quantile(.9)
print(f"\nTop 10% of clinicians scored {human_benchmark:.1f} or higher")

vec_A = (df.loc[df['condition']=='cognitive_layer','score']> human_benchmark).mean()
vec_B = (df.loc[df['condition']=='standalone_llms','score']> human_benchmark).mean()
print(f"{vec_A:.1%} of all cognitive layer conversations scored higher than top 10% of clinicians")
print(f"{vec_B:.1%} of all standalone LLM conversations scored higher than top 10% of clinicians")

# ------------------------------------------------------------------------------------------------------------------------------------
# Plot
# ------------------------------------------------------------------------------------------------------------------------------------
summary = (
    df
    .groupby(['model','condition'])['score']
    .agg(['mean','sem'])
    .reset_index()
)

model_order = ['claude','llama','gemini','gpt4','']
condition_order = CONDITION_ORDER

fig = px.bar(
    summary,
    x='model',
    y='mean',
    error_y='sem',
    color='condition',
    color_discrete_map=COLOURS,
    category_orders={'model': model_order, 'condition': condition_order},
    width=500,
    height=400,
    template='simple_white',
    title='CTRS Score',
    barmode='group',
    labels={'mean': 'Score', 'model': 'Model'}
)
fig.update_layout(font={'family': 'Arial'})
fig.update_yaxes(range=[0,6])
fig.show()

N = 227
OVERALL CTRS:


,mean,std
condition,,
cognitive_layer,4.53,1.11
human_therapist,2.53,0.94
standalone_llms,3.16,0.93


BELIEVED HUMAN:


,mean,std
condition,,
cognitive_layer,46.5%,30.0%
human_therapist,67.3%,30.6%
standalone_llms,46.2%,30.4%


Shapiro-Wilk: 0.990, p = 0.16
Kolmogorov-Smirnov: 0.040, p = 0.88
OVERALL CTRS: 2 X 4 MODEL
                                          sum_sq     df           F  \
C(model)                                1.444699    3.0    0.546294   
C(condition)                           92.448977    1.0  104.875031   
C(model):C(condition)                   0.802078    3.0    0.303295   
believed_human                         34.723835    1.0   39.391061   
C(model):believed_human                 4.558972    3.0    1.723914   
C(condition):believed_human             1.674042    1.0    1.899050   
C(model):C(condition):believed_human    2.169682    3.0    0.820436   
Residual                              163.080387  185.0         NaN   

                                            PR(>F)            p  partial_n2  
C(model)                              6.512411e-01         0.65    0.008781  
C(condition)                          8.778420e-20  8.78e-20***    0.361794  
C(model):C(condition)             

Shapiro-Wilk: 0.995, p = 0.63
Kolmogorov-Smirnov: 0.047, p = 0.68
OVERALL CTRS: 3-WAY MODEL
                                 sum_sq     df          F        PR(>F)  \
C(condition)                 151.155521    2.0  87.652948  9.396632e-29   
believed_human                39.305225    1.0  45.585087  1.271344e-10   
C(condition):believed_human    1.808353    2.0   1.048639  3.521514e-01   
Residual                     190.554744  221.0        NaN           NaN   

                                       p  partial_n2  
C(condition)                 9.40e-29***    0.442350  
believed_human               1.27e-10***    0.170996  
C(condition):believed_human         0.35    0.009401  
Residual                             nan         NaN  
HUMAN BENCHMARKING
Cognitive Layer - Human Therapists = 2.0
                             Contrast  Adj. mean diff                     SE  \
0  cognitive layer vs standalone LLMs       -1.087072  [0.24084461408972174]   
1  cognitive layer vs human therapist

#### Supplementary: Controlling for latency and session duration

In [29]:
# ------------------------------------------------------------------------------------------------------------------------------------
# Supplementary: Controlling for latency and session duration
# ------------------------------------------------------------------------------------------------------------------------------------
df = df.merge(df_pids[['pid','mean_response_latency_seconds','session_duration_minutes']], on='pid', how='left')

# Run models
# --- 2 x 4 model
model_df = df.loc[df['condition']!='human_therapist',].copy().reset_index(drop=True)

model_fit = smf.ols('score ~ C(model) * C(condition) * believed_human + mean_response_latency_seconds + session_duration_minutes', data=model_df).fit()
anova_table = format_anova_table(model_fit)

print_title('Overall CTRS: 2 x 4 model')
print(anova_table)

# --- 3-way model
model_df = df.groupby(['pid','condition','believed_human'])[['score','mean_response_latency_seconds','session_duration_minutes']].mean().reset_index()

model_fit = smf.ols('score ~ C(condition) * believed_human + mean_response_latency_seconds + session_duration_minutes', data=model_df).fit()
anova_table = format_anova_table(model_fit)

print_title('Overall CTRS: 3-way model')
print(anova_table)

OVERALL CTRS: 2 X 4 MODEL
                                          sum_sq     df          F  \
C(model)                                0.306723    3.0   0.116217   
C(condition)                           20.864757    1.0  23.716809   
C(model):C(condition)                   1.750852    3.0   0.663393   
believed_human                         33.103106    1.0  37.628047   
C(model):believed_human                 4.262517    3.0   1.615057   
C(condition):believed_human             1.786407    1.0   2.030595   
C(model):C(condition):believed_human    1.941722    3.0   0.735714   
mean_response_latency_seconds           1.912116    1.0   2.173488   
session_duration_minutes                0.032370    1.0   0.036794   
Residual                              160.993432  183.0        NaN   

                                            PR(>F)            p  partial_n2  
C(model)                              9.505112e-01         0.95    0.001902  
C(condition)                          2.405065e

#### Inter-rater reliability

In [ ]:
df = (
    df_ctrs
    .groupby('pid')[['human1_score','human2_score']]
    .mean()
    .reset_index()
    .dropna()
    .reset_index(drop=True)
)

print(f"N = {len(df)}")

compute_all_icc(df[['human1_score','human2_score']])

stat = sp.spearmanr(df[['human1_score','human2_score']])
print(f"Spearman's rho = {stat.correlation:.2f}, p = {readable_pvalue(stat.pvalue)}")


N = 214
Intraclass Correlation Coefficients (ICC) between human and classifier:
ICC(1,1): 0.405
ICC(1,k): 0.577
ICC(2,1): 0.405
ICC(2,k): 0.576
ICC(3,1): 0.404
ICC(3,k): 0.576
Spearman's rho = 0.41, p = 3.92e-10***


### CTRS subscales

In [22]:
df = (
        df_ctrs[['pid','item','human1_score']]
        .copy()
        .rename(columns={'human1_score':'score'})
    )

# Add transcript metadata
df = df.merge(df_pids[['pid','condition','model']], on='pid', how='left')

# Mann Whitney U tests
items = list(df['item'].unique())
n_items = len(items)
max_pval = 0

results = []
for item in items:
    item_index = df['item']==item
    item_results = {'item': item}
    
    for comparison in [['cognitive_layer','standalone_llms'],['cognitive_layer','human_therapist']]:
        vec_A = df.loc[(df['condition']==comparison[0]) & item_index,'score']
        vec_B = df.loc[(df['condition']==comparison[1]) & item_index,'score']
        stat = sp.mannwhitneyu(vec_A,vec_B)
        p = stat.pvalue * 2 * n_items # Bonferroni correction
        if p > max_pval:
            max_pval = p
        
        comparison_key = f"{comparison[0]}_vs_{comparison[1]}"
        item_results[comparison_key.capitalize().replace('_',' ')] = f"U = {stat.statistic}, p = {readable_pvalue(p)}"
    
    results.append(item_results)

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))
print(f"\nMaximum p-value = {readable_pvalue(max_pval)}")

# Plot
plot_df = df.groupby(['item','condition'])['score'].agg(['mean','sem']).reset_index()

item_order = plot_df.loc[plot_df['condition']=='cognitive_layer',].sort_values('mean',ascending=False)['item'].unique().tolist()

fig = px.bar(
    plot_df,
    x='item',
    y='mean',
    error_y='sem',
    color='condition',
    color_discrete_map=COLOURS,
    category_orders={'condition': CONDITION_ORDER,'item': item_order},
    width=600,
    height=400,
    template='simple_white',
    title='CTRS Subscales',
    barmode='group'
)
# fig.update_traces(width=0.1)
fig.update_yaxes(range=[0,6])
fig.update_xaxes(tickvals=[x for x in range(0,len(item_order))],ticktext=item_order)
fig.update_layout(font={'family': 'Arial'})
fig.show()

         item Cognitive layer vs standalone llms Cognitive layer vs human therapist
       agenda        U = 9387.0, p = 2.47e-25***        U = 2468.0, p = 1.53e-11***
     feedback        U = 8076.0, p = 1.51e-12***        U = 2326.5, p = 1.13e-08***
understanding        U = 7557.0, p = 1.10e-08***        U = 2050.0, p = 1.31e-04***
interpersonal        U = 7266.5, p = 7.29e-07***        U = 1982.0, p = 8.63e-04***
collaboration        U = 7382.5, p = 1.37e-07***        U = 2121.0, p = 1.69e-05***
       pacing        U = 7752.0, p = 4.22e-10***        U = 2415.0, p = 3.72e-10***
    discovery        U = 7334.5, p = 3.02e-07***        U = 2107.5, p = 2.76e-05***
        focus        U = 7355.5, p = 1.72e-07***        U = 2203.0, p = 1.08e-06***
     strategy        U = 7057.0, p = 1.27e-05***        U = 2083.5, p = 5.39e-05***
    technique        U = 7200.0, p = 2.19e-06***        U = 2219.0, p = 7.42e-07***
     homework        U = 7402.0, p = 1.52e-07***        U = 2290.0, p = 7.14

## General Clinical Practice

### Dimensions

In [23]:
df = (
    df_general
    .copy()
    .merge(df_pids[['pid','condition','model']], on='pid', how='left')
)

items = ['safety','structure','rationale','interpersonal','omission','trust']

df = df.loc[df['item'].isin(items),].reset_index(drop=True)

df['score'] = df['score'].replace({
    'Strong disagree': 0,
    'Disagree': 1,
    'Neutral': 2,
    'Agree': 3,
    'Strong agree': 4
}).astype(int)

# Reverse code safety & omission
mask = df['item'].isin(['safety','omission'])
df.loc[mask,'score'] = 4 - df.loc[mask,'score']

# Stats
stats = pd.DataFrame({'item': [x for x in df['item'].unique() if x!='Was human']})
for item in stats['item']:
    subset = df.loc[df['item']==item,].copy()

    vec_A = subset.loc[subset['condition']=='cognitive_layer','score'].dropna().astype(int)
    vec_B = subset.loc[subset['condition']=='standalone_llms','score'].dropna().astype(int)
    vec_C = subset.loc[subset['condition']=='human_therapist','score'].dropna().astype(int)

    stat = sp.mannwhitneyu(vec_A,vec_B)
    stats.loc[stats['item']==item,'U_cl_vs_standalone'] = stat.statistic
    stats.loc[stats['item']==item,'p_cl_vs_standalone'] = stat.pvalue

    stat = sp.mannwhitneyu(vec_A,vec_C)
    stats.loc[stats['item']==item,'U_cl_vs_human'] = stat.statistic
    stats.loc[stats['item']==item,'p_cl_vs_human'] = stat.pvalue

stats['bonf_cl_vs_standalone'] = stats['p_cl_vs_standalone']*12
stats['bonf_cl_vs_human'] = stats['p_cl_vs_human']*12

for col in ['p_cl_vs_standalone','bonf_cl_vs_standalone','p_cl_vs_human','bonf_cl_vs_human']:
    stats[col] = stats[col].apply(readable_pvalue)

display(stats)

# Plot for main text
summary = df.groupby(['item','condition'])['score'].agg(['mean','sem']).reset_index()
summary['item'] = summary['item'].str.capitalize()

fig = px.bar(
    summary,
    x='item',
    y='mean',
    error_y='sem',
    color='condition',
    title='General Clinical Performance',
    color_discrete_map=COLOURS,
    category_orders={'item': [x.capitalize() for x in items], 'condition': CONDITION_ORDER},
    width=500,
    height=400,
    labels={'mean': 'Score', 'item': 'Scale'},
    barmode='group',
    template='simple_white'
)
fig.update_layout(font={'family': 'Arial'})
fig.update_yaxes(range=[0,4])
fig.show()

# Plot for supplementary
summary = (
    df
    .loc[df['condition']!='human_therapist']
    .groupby(['item','condition','model'])['score']
    .agg(['mean','sem'])
    .reset_index()
)
summary['item'] = summary['item'].str.capitalize()

fig = px.bar(
    summary,
    x='item',
    y='mean',
    facet_col='model',
    facet_col_wrap=2,
    error_y='sem',
    color='condition',
    title='General Clinical Performance - Per LLM',
    color_discrete_map=COLOURS,
    category_orders={'item': [x.capitalize() for x in items], 'condition': CONDITION_ORDER},
    width=700,
    height=600,
    labels={'mean': 'Score', 'item': 'Scale'},
    barmode='group',
    template='simple_white'
)
fig.update_layout(font={'family': 'Arial'})
fig.update_yaxes(range=[0,4])
fig.for_each_annotation(lambda x: x.update(text=x.text.split('=')[-1]))
fig.show()

,item,U_cl_vs_standalone,p_cl_vs_standalone,U_cl_vs_human,p_cl_vs_human,bonf_cl_vs_standalone,bonf_cl_vs_human
0,interpersonal,6863.5,3.35e-06***,1997.0,1.31e-05***,4.02e-05***,1.57e-04***
1,structure,7313.5,1.58e-09***,2226.5,3.94e-09***,1.89e-08***,4.73e-08***
2,safety,5905.5,0.026*,1781.0,0.003**,0.31,0.031*
3,trust,7010.0,7.90e-07***,1998.0,2.13e-05***,9.48e-06***,2.55e-04***
4,rationale,7354.0,2.32e-09***,2066.0,1.49e-06***,2.79e-08***,1.79e-05***
5,omission,7004.0,7.34e-07***,2078.0,1.81e-06***,8.81e-06***,2.17e-05***


### Pairwise comparisons

In [24]:
df = df_comparisons.copy()
for i in [1,2]:
    df[f"condition{i}"] = df[f"pid{i}"].map(df_pids.set_index('pid')['condition'])
    df[f"model{i}"] = df[f"pid{i}"].map(df_pids.set_index('pid')['model'])
df['winner'] = df['choice'].map(df_pids.set_index('pid')['condition'])

# Summarise preferences per item
summary = (
    df
    .groupby(['item'])['winner']
    .agg(lambda x: (x=='cognitive_layer').mean())
    .reset_index()
    .rename(columns={'winner':'proportion'})
    .assign(winner='cognitive_layer')
)

insert = summary.copy()
insert['winner'] = 'standalone_llms'
insert['proportion'] = 1 - summary['proportion']

summary = pd.concat([
    summary,
    insert
]).reset_index(drop=True)

summary['label'] = summary['proportion'].apply(lambda x: f"{x:.0%}")
summary.loc[summary['winner']=='standalone_llms','label'] = ''

display(summary.loc[summary['winner']=='cognitive_layer',])

items = ['structure','preference','accuracy','trust','interpersonal','focus','safety']

# Calculate standard deviation for proportions using the formula: sqrt(p * (1-p) / n)
# where p is the proportion and n is the sample size
n = len(df)
p = (df['winner']=='cognitive_layer').mean()
std_error = np.sqrt(p * (1-p) / n)
print(f"Average preference rate = {p:.1%} ± {std_error:.1%}")

# statistically compare each subscale to chance (50%)
def stats_pairwise_comparisons(df,items):
    n_items = len(items)
    max_pval = 0

    results = []
    for item in items:
        stat = sp.binomtest(
            ((df['item']==item) & (df['winner']=='cognitive_layer')).sum(),
            ((df['item']==item)).sum(),
            p=.5
        )
        pval = stat.pvalue
        bpval = n_items * pval
        results.append({
            'item': item,
            'p': readable_pvalue(stat.pvalue),
            'p(bonferroni)': readable_pvalue(bpval)
        })
        max_pval = max(max_pval, bpval)

    results_df = pd.DataFrame(results)
    display(results_df)

    print(f"\nMaximum p-value = {readable_pvalue(max_pval)}")

    return results_df

stats_pairwise_comparisons(df,items)

# Plot
item_labels = {
    'structure': 'Session Structure',
    'preference': 'Better Clinical Practice',
    'accuracy': 'Therapy Accuracy',
    'trust': 'Real-World Adequacy',
    'interpersonal': 'Interpersonal Skills',
    'focus': 'Focus on Problem',
    'safety': 'Harm Avoidance'
}

def plot_pairwise_comparisons(summary,items,title='Pairwise comparisons'):
    fig = px.bar(
        summary,
        y='item',
        x='proportion',
        color='winner',
        color_discrete_map=COLOURS,
        title=title,
        width=350,
        height=400,
        barmode='stack',
        category_orders={'item': items},
        labels={'item': 'Criteria', 'winner': 'Preference', 'proportion': 'Proportion'},
        orientation='h',
        template='simple_white',
        text='label',
    )
    fig.update_xaxes(tickformat=".0%",range=[0,1])
    fig.update_layout(font={'family': 'Arial'})
    fig.add_vline(x=.5,line_dash='dot',line_color='black',line_width=1,opacity=1,layer='above')
    fig.show()
    return fig

fig = plot_pairwise_comparisons(summary,items)
fig.write_image('../results/pairwise_comparisons.svg',width=350,height=400)

# Plot per LLM
for model in ['claude','llama','gemini','gpt4']:

    print_title(f'Pairwise comparisons: {model}')

    summary = (
        df
        .loc[df['model1']==model,]
        .groupby(['item'])['winner']
        .agg(lambda x: (x=='cognitive_layer').mean())
        .reset_index()
        .rename(columns={'winner':'proportion'})
        .assign(winner='cognitive_layer')
    )

    insert = summary.copy()
    insert['winner'] = 'standalone_llms'
    insert['proportion'] = 1 - summary['proportion']

    summary = pd.concat([
        summary,
        insert
    ]).reset_index(drop=True)

    summary['label'] = summary['proportion'].apply(lambda x: f"{x:.0%}")
    summary.loc[summary['winner']=='standalone_llms','label'] = ''

    stats_pairwise_comparisons(df.loc[df['model1']==model,],items)

    fig = plot_pairwise_comparisons(summary,items,title=f'Pairwise comparisons: {model}')
    fig.write_image(f'../results/pairwise_comparisons_{model}.svg',width=350,height=400)

,item,proportion,winner,label
0,accuracy,0.854167,cognitive_layer,85%
1,focus,0.791667,cognitive_layer,79%
2,interpersonal,0.812500,cognitive_layer,81%
3,preference,0.864583,cognitive_layer,86%
4,safety,0.770833,cognitive_layer,77%
5,structure,0.875000,cognitive_layer,88%
6,trust,0.822917,cognitive_layer,82%


Average preference rate = 82.7% ± 1.5%


,item,p,p(bonferroni)
0,structure,1.83e-14***,1.28e-13***
1,preference,1.20e-13***,8.41e-13***
2,accuracy,7.24e-13***,5.07e-12***
3,trust,9.94e-11***,6.96e-10***
4,interpersonal,4.45e-10***,3.11e-09***
5,focus,7.32e-09***,5.12e-08***
6,safety,9.44e-08***,6.61e-07***



Maximum p-value = 6.61e-07***


PAIRWISE COMPARISONS: CLAUDE


...

,item,p,p(bonferroni)
0,structure,3.59e-05***,2.51e-04***
1,preference,3.59e-05***,2.51e-04***
2,accuracy,2.77e-04***,0.002**
3,trust,1.19e-07***,8.34e-07***
4,interpersonal,2.98e-06***,2.09e-05***
5,focus,2.77e-04***,0.002**
6,safety,2.77e-04***,0.002**



Maximum p-value = 0.002**


PAIRWISE COMPARISONS: GPT4


,item,p,p(bonferroni)
0,structure,5.34e-04***,0.004**
1,preference,5.34e-04***,0.004**
2,accuracy,0.002**,0.017*
3,trust,0.029*,0.20
4,interpersonal,0.002**,0.017*
5,focus,0.08,0.53
6,safety,0.17,> .999



Maximum p-value = > .999
